## AgentCore Evaluations - evaluator 생성

이 튜토리얼에서는 AgentCore Evaluations의 기본 제공 지표와 사용자 지정 지표를 알아봅니다.
각 유형을 언제 사용해야 하는지, 특정 요구 사항에 맞는 사용자 지정 evaluator를 만드는 방법을 배웁니다.

### 학습 내용
- 기본 제공 evaluator와 사용 사례 이해
- 특수한 요구 사항을 위한 사용자 지정 evaluator 생성
- 에이전트에 적합한 평가 방식 선택

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                      |
|:--------------------|:-------------------------------------------------------------------------------|
| 튜토리얼 유형       | 사용자 지정 evaluator 지표 생성                                               |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                     |
| 튜토리얼 구성 요소  | 기본 제공 evaluator 조회, 사용자 지정 evaluator 지표 생성                     |
| 튜토리얼 분야       | 여러 분야 공통                                                                 |
| 예제 난이도         | 쉬움                                                                           |
| 사용 SDK            | Amazon Bedrock AgentCore Starter Toolkit                                       |

## Evaluator 유형

### 기본 제공 evaluator

AgentCore는 Large Language Model(LLM)을 판정자로 사용해 에이전트 성능을 평가하는 사전 구성된 evaluator 13개를 제공합니다. 이러한 evaluator에는 세심하게 작성된 prompt 템플릿과 미리 선택된 evaluator 모델이 포함되어 있어 여러 사용 사례에서 평가 기준을 표준화할 수 있습니다. 바로 사용할 수 있으므로 시작할 때 별도의 구성을 추가할 필요가 없습니다.

이러한 evaluator는 다음 네 가지 그룹으로 나뉩니다.

- **응답 품질**: 각 턴에서 에이전트가 예상대로 작동하는지 판단하는 데 도움이 되는 evaluator입니다. trace 단위로 작동하며 각 사용자-에이전트 상호 작용을 평가합니다.
- **작업 완료**: 세션 전체를 평가하는 evaluator인 Goal success가 이 범주에 속합니다. 멀티턴 대화에서 이 evaluator는 사용자의 목표가 완료되었는지와 최종 결과가 달성되었는지를 판단하는 데 도움을 줍니다. 에이전트가 후속 질문을 하는 상황에서는 요청한 작업이 실제로 완료되었는지 파악하는 데 꼭 필요합니다.
- **도구 수준**: 도구 호출이 얼마나 성공적인지 파악하는 데 도움이 되는 evaluator입니다. 도구 수준에서 에이전트의 도구 및 파라미터 선택 정확도를 측정합니다. 에이전트가 한 턴에 서로 다른 도구를 두 개 이상 호출하면 각 도구에 해당하는 지표가 trace에 개별적으로 기록됩니다.
- **안전성**: 에이전트가 유해한 응답을 생성하거나 개인 또는 집단에 대해 고정관념에 기반한 일반화를 하는지 감지하는 evaluator입니다.

기본 제공 지표를 사용하면 prompt부터 모델까지 모든 요소가 자동으로 처리됩니다. 모든 사용자에게 일관되고 신뢰할 수 있는 평가를 제공하기 위해 해당 evaluator는 수정할 수 없습니다. 다만 기본 제공 지표를 기반으로 자체 지표를 만들 수 있으며, 이를 위해 기본 제공 evaluator의 **Prompt Templates**를 제공합니다.

### 사용자 지정 evaluator

사용자 지정 evaluator를 사용하면 LLM을 기반 판정자로 활용하면서 평가 프로세스의 모든 요소를 정의할 수 있어 유연성이 극대화됩니다. 사용자 지정 evaluator에서는 다음 항목을 조정할 수 있습니다.

- **Evaluator 모델**: 평가 요구 사항에 가장 적합한 LLM 선택
- **평가 prompt**: 사용 사례에 맞는 평가 지침 작성
- **점수 체계**: 조직의 지표에 부합하는 점수 시스템 설계

### 에이전트에서 AgentCore Observability trace 생성

AgentCore Observability는 [OpenTelemetry (OTEL)](https://opentelemetry.io/) trace를 상세 실행 데이터 수집 및 구조화의 기반으로 활용해 호출 중 에이전트 동작을 포괄적으로 보여 줍니다. AgentCore는 [AWS Distro for OpenTelemetry (ADOT)](https://aws-otel.github.io/)를 사용해 다양한 에이전트 프레임워크의 여러 OTEL trace 유형을 계측합니다.

이 튜토리얼의 에이전트처럼 AgentCore Runtime에서 에이전트를 호스팅하면 최소한의 구성만으로 AgentCore Observability 계측이 자동 적용됩니다. `requirements.txt`에 `aws-opentelemetry-distro`를 포함하면 AgentCore Runtime이 OTEL 구성을 자동으로 처리합니다. 에이전트가 AgentCore Runtime에서 실행되지 않는 경우 AgentCore Observability에서 사용할 수 있도록 ADOT로 계측해야 합니다. telemetry 데이터를 CloudWatch로 보내도록 환경 변수를 구성하고 OpenTelemetry 계측을 적용해 에이전트를 실행해야 합니다.

프로세스는 다음과 같습니다.

![세션 trace](../images/observability_traces.png)

세션 trace가 AgentCore Observability에 제공되면 AgentCore Evaluations를 사용해 에이전트 동작을 평가할 수 있습니다.

### 평가 수준
AgentCore Evaluations는 에이전트 상호 작용의 여러 수준에서 작동합니다. 세션 정보를 사용해 주고받은 대화 전체를 분석할 수 있고, trace 정보를 사용해 대화의 개별 턴에서 사용자 질문에 대한 에이전트 응답을 평가할 수도 있습니다. 또한 span 데이터를 사용해 도구 호출과 파라미터 선택을 포함한 턴 내부 정보를 평가할 수 있습니다.

각 수준에 맞는 사용자 지정 지표를 만들 수 있습니다. 기본 제공 지표는 다음 범위에서 작동합니다.

![평가 지표 수준](../images/metrics_per_level.png)

이 튜토리얼에서는 TRACE 수준의 지표를 생성합니다.

### 튜토리얼 결과

이 튜토리얼을 마치면 AgentCore Evaluations의 기본 제공 지표와 사용자 지정 지표를 이해하고, 에이전트의 응답 품질을 측정하는 사용자 지정 지표를 생성하게 됩니다. 


### 사전 요구 사항
이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS 자격 증명
* Amazon Bedrock AgentCore Starter toolkit

### AgentCore Evaluations 사용

Amazon Bedrock AgentCore는 에이전트와 도구를 개발, 배포 및 모니터링하기 위한 다양한 인터페이스를 지원합니다.

세부 요소를 완전히 제어하려면 [control plane](https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/Welcome.html) 및 [data plane](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/Welcome.html) API를 사용할 수 있습니다. 이러한 API는 AWS SDK([boto3](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/quickstart.html), [AWS SDK for Java](https://docs.aws.amazon.com/sdk-for-java/latest/developer-guide/home.html), [AWS SDK for JavaScript](https://docs.aws.amazon.com/sdk-for-javascript/v3/developer-guide/welcome.html))와 AWS Command Line Interface([AWS CLI](https://aws.amazon.com/cli/)) 같은 AWS 개발자 도구를 통해 제공됩니다.

더 간단한 방식으로는 [AgentCore Python SDK](https://github.com/aws/bedrock-agentcore-sdk-python)와 [AgentCore Starter Toolkit](https://github.com/aws/bedrock-agentcore-starter-toolkit)을 사용할 수 있습니다. AgentCore Python SDK는 에이전트 개발을 위한 Python 기본 요소를 제공하고, AgentCore Starter Toolkit은 AgentCore 기능을 위한 CLI 도구와 상위 수준 추상화를 제공합니다.

![AgentCore 인터페이스](../images/agentcore_interfaces.png)

이 튜토리얼에서는 간편하게 시작할 수 있도록 AgentCore Starter Toolkit을 사용합니다. `03-advanced` 폴더에서 boto3를 직접 사용하는 예제를 확인할 수 있습니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation
import json
from boto3.session import Session

In [ ]:
boto_session = Session()
region = boto_session.region_name
print(region)

### AgentCore Evaluations 클라이언트 초기화

이제 평가 클라이언트를 초기화하겠습니다. 이 튜토리얼에서는 AgentCore 구성 요소와의 상호 작용을 단순화해 빠르게 시작할 수 있도록 돕는 추상화 SDK인 [AgentCore Starter Toolkit](https://github.com/aws/bedrock-agentcore-starter-toolkit)을 사용합니다. 

In [ ]:
eval_client = Evaluation(region=region)

### 기본 제공 evaluator 조회

사용 가능한 기본 제공 evaluator를 조회해 각각 어디에 사용할 수 있는지 알아보겠습니다. `list_evaluators()` 함수를 사용하면 됩니다. 

In [ ]:
available_evaluators = eval_client.list_evaluators()
available_evaluators

나중에 온디맨드 및 온라인 평가에서 사용할 수 있도록 정보를 딕셔너리로 가져올 수도 있습니다.

In [ ]:
print(
    available_evaluators["evaluators"][0]["evaluatorId"],
    available_evaluators["evaluators"][0]["description"],
)

`Builtin.Correctness` 지표는 응답 품질을 평가하는 데 도움이 됩니다. 세부 내용을 이해하기 위해 이 지표를 자세히 살펴보겠습니다. 여기에는 `get_evaluator` 메서드를 사용할 수 있습니다.

In [ ]:
eval_client.get_evaluator(evaluator_id="Builtin.Correctness")

이 evaluator는 응답을 Incorrect, Partially Correct, Correct의 세 수준으로 분류합니다. 이 사용 사례에서는 더 세분화된 5단계 척도를 사용하려고 하므로 사용자 지정 evaluator를 생성합니다.

### 사용자 지정 evaluator 생성

이제 5단계 척도를 사용하는 응답 품질용 사용자 지정 지표를 생성하겠습니다. evaluator 모델을 선택하고 평가 지침을 제공한 다음 평점 척도를 설정해야 합니다. 여기서는 Very Good부터 Very Poor까지의 척도를 사용합니다. JSON 파일에서 평가 구성을 가져오겠습니다.

In [ ]:
with open("metric.json") as f:
    print("Reading custom metric details")
    eval_config = json.load(f)
eval_config

그런 다음 `create_evaluator` 메서드로 evaluator를 생성할 수 있습니다. `TRACE` 수준에 `response_Quality` evaluator를 생성하겠습니다. 

사용자 지정 evaluator를 생성할 때 도구 호출, trace 또는 세션 수준 중 적용할 수준을 선택할 수 있습니다. 

* **도구 호출**은 에이전트가 외부 함수, API 또는 기능을 호출한 것을 나타내는 span입니다. 도구 호출 span은 일반적으로 도구 이름, 입력 파라미터, 실행 시간 및 출력 등의 정보를 수집합니다. 도구 호출 세부 정보는 에이전트가 도구를 올바르고 효율적으로 선택하고 사용했는지 평가하는 데 사용됩니다.

* **Trace**는 단일 에이전트 실행 또는 요청에 관한 전체 기록입니다. trace에는 해당 실행 중 수행된 개별 작업을 나타내는 span이 하나 이상 포함됩니다. trace는 에이전트의 의사 결정과 도구 사용을 엔드 투 엔드로 보여 줍니다.

* **세션**은 단일 사용자 또는 워크플로에서 발생한 관련 상호 작용의 논리적 그룹을 나타냅니다. 세션에는 trace가 하나 이상 포함될 수 있습니다. 세션을 사용하면 개별 요청에만 초점을 맞추지 않고 여러 단계의 상호 작용 전반에서 에이전트 동작을 확인하고 평가할 수 있습니다. 

In [ ]:
custom_evaluator = eval_client.create_evaluator(
    name="response_quality_for_scope",
    level="TRACE",
    description="Response quality evaluator",
    config=eval_config,
)

### 다음 튜토리얼을 위한 evaluator 정보 저장

다음 튜토리얼에서 사용할 evaluatorId를 저장하겠습니다. 이를 위해 `evaluator_id` 변수를 저장합니다.

In [ ]:
evaluator_id = custom_evaluator["evaluatorId"]

In [ ]:
%store evaluator_id

#### 축하합니다

다음 튜토리얼에서 사용할 사용자 지정 evaluator를 생성했습니다.